In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

# Initialize Spark session
spark = SparkSession.builder.appName("BroadcastVariableExample").getOrCreate()




In [0]:
# Create a dictionary to broadcast (lookup table)
state_mapping = {
    "CA": "California",
    "NY": "New York",
    "TX": "Texas",
    "FL": "Florida"
}

# Broadcast the dictionary
broadcast_var = spark.sparkContext.broadcast(state_mapping)



---------------------------------------------------------------------------
PySparkAttributeError                     Traceback (most recent call last)
File <command-7279907739582862>, line 10
      2 state_mapping = {
      3     "CA": "California",
      4     "NY": "New York",
      5     "TX": "Texas",
      6     "FL": "Florida"
      7 }
      9 # Broadcast the dictionary
---> 10 broadcast_var = spark.sparkContext.broadcast(state_mapping)

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/session.py:1108, in SparkSession.__getattr__(self, name)
   1106 def __getattr__(self, name: str) -> Any:
   1107     if name in ["_jsc", "_jconf", "_jvm", "_jsparkSession", "sparkContext", "newSession"]:
-> 1108         raise PySparkAttributeError(
   1109             errorClass="JVM_ATTRIBUTE_NOT_SUPPORTED", messageParameters={"attr_name": name}
   1110         )
   1111     return object.__getattribute__(self, name)

PySparkAttributeError: [JVM_ATTRIBUTE_NOT_SUPPORTED] 

In [0]:
# Create a DataFrame with state codes
data = [(1, "CA"), (2, "NY"), (3, "TX"), (4, "FL")]
df = spark.createDataFrame(data, ["id", "state_code"])



In [0]:
# Define a UDF that uses the broadcast variable
def map_state(code):
    return broadcast_var.value.get(code, "Unknown")

map_state_udf = udf(map_state, StringType())

# Apply the UDF to transform the DataFrame
result_df = df.withColumn("state_name", map_state_udf(df.state_code))

# Show results
result_df.show()